In [1]:
#!pip install pandas numpy matplotlib missingno

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import missingno as msno

import os
from pathlib import Path

#import sklearn.preprocessing

pd.options.display.max_rows=1000

print("Imported libraries sucessfully! yay")

Imported libraries sucessfully! yay


In [2]:
def find_project_root(
        start: Path = Path.cwd()
        ,project_name: str = "Project_SupervisedLearning_1"
) -> Path:
    """
    Find a project directory containing both data/ and notebooks/ folders.

    Searches:
    1. current dir
    2. current dir / Project_1
    3. parent dirs
    """

    start = start.resolve()

    candidates = [
        start
        ,start / project_name
        ,*start.parents
    ]

    #print("Candidates to search: ", candidates)

    checked = set() #unique set to store checked candidates

    for candidate in candidates:
        candidate = candidate.resolve()
        #print("Check candidate: ", candidate)
        if candidate in checked:
            #print("Already checked! Skip to next candidate")
            continue

        checked.add(candidate)

        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            #print("Found ", candidate)
            return candidate

    raise FileNotFoundError(
        f"Could not find {project_name!r} containing data/ and notebooks/."
        "Set PROJECT_ROOT manually using Path()"
    )

In [3]:
PROJECT_ROOT = find_project_root()
print(PROJECT_ROOT)

TRAIN_DIR = PROJECT_ROOT / "data" / "train"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

print("Train Data directory: ", TRAIN_DIR)


C:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\Project_1_SupervisedLearning
Train Data directory:  C:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\Project_1_SupervisedLearning\data\train


In [4]:
def find_unique_file(directory: Path, filename: str) -> Path:
    matches = [path for path in directory.rglob(filename)
                      if path.is_file()]
    print("Matched files: ",matches)
    if not matches:
        raise FileNotFoundError(f"Could not find {filename!r} under {directory}")

    if len(matches) > 1:
        formatted = "\n".join(str(path) for path in matches)
        print("Different formats of dup matches: ", formatted)
        raise RuntimeError(
            f"Found multiple copies of {filename!r}:\n{formatted}"
        )

    return matches[0]

In [5]:
def load_data(train_dir: Path):
    PRICE_PATH = find_unique_file(directory=train_dir, filename="stock_prices.csv")
    STOCK_LIST_PATH = find_unique_file(train_dir, "stock_list.csv")
    FINANCIALS_PATH = find_unique_file(train_dir, "financials.csv")

    prices = pd.read_csv(PRICE_PATH, dtype={'SecuritiesCode':'string'} , low_memory=False)
    stock_list = pd.read_csv(STOCK_LIST_PATH, dtype={'SecuritiesCode':'string'}, low_memory=False)
    financials = pd.read_csv(FINANCIALS_PATH, dtype={'SecuritiesCode':'string'}, low_memory=False)

    #standardize main identifier (primary key)
    for name, frame in {
        "prices": prices
        ,"stock_list": stock_list
        ,"financials": financials
    }.items():
        if "SecuritiesCode" not in frame.columns:
            raise ValueError(f"{name} has no SecuritiesCode column")
        
        frame['SecuritiesCode'] =frame['SecuritiesCode'].str.strip()
        
        missing_count = frame['SecuritiesCode'].isna().sum()
        
        print(f"{name}: {missing_count} rows with missing SecuritiesCode")
        
        '''
        if frame['SecuritiesCode'].isna().any():
                    raise ValueError(f"{name}.SecuritiesCode contains missing values.")
        invalid_codes = ~frame['SecuritiesCode'].str.fullmatch(rf"\d{4}")
        
        if invalid_codes.any():
            raise ValueError(f"{name} contains invalid SecuritiesCode values (not 4 digit code).") 
        '''
     
    
    prices['Date'] = pd.to_datetime(prices['Date'], errors='raise')
    
    for column in ['Date'
                   ,'DisclosedDate'
                   ,'CurrentPeriodEndDate'
                   ,'CurrentFiscalYearStartDate'
                   ,'CurrentFiscalYearEndDate']:
        if column in financials.columns:
            financials[column] = pd.to_datetime(financials[column], errors='coerce')
        
    for column in ['EffectiveDate'
                ,'TradeDate']:
        if column in stock_list.columns:
            stock_list[column] = pd.to_datetime(stock_list[column], errors='coerce')
        
    prices = prices.sort_values(['Date','SecuritiesCode']).reset_index(drop=True)
    return prices, stock_list, financials      

<>:28: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:28: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\MY NGOC\AppData\Local\Temp\ipykernel_4696\1611003611.py:28: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  invalid_codes = ~frame['SecuritiesCode'].str.fullmatch(rf"\d{4}")


In [6]:
prices_raw, stock_list_raw, financials_raw = load_data(train_dir=TRAIN_DIR)

Matched files:  [WindowsPath('C:/Users/MY NGOC/Documents/MAU_TACAS_68803/DA631E_ArtificialIntelligenceForDataScience/Project_1_SupervisedLearning/data/train/stock_prices.csv')]
Matched files:  [WindowsPath('C:/Users/MY NGOC/Documents/MAU_TACAS_68803/DA631E_ArtificialIntelligenceForDataScience/Project_1_SupervisedLearning/data/train/stock_list.csv')]
Matched files:  [WindowsPath('C:/Users/MY NGOC/Documents/MAU_TACAS_68803/DA631E_ArtificialIntelligenceForDataScience/Project_1_SupervisedLearning/data/train/financials.csv')]
prices: 0 rows with missing SecuritiesCode
stock_list: 0 rows with missing SecuritiesCode
financials: 2 rows with missing SecuritiesCode


In [7]:
financials_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 92956 entries, 0 to 92955
Data columns (total 45 columns):
 #   Column                                                                        Non-Null Count  Dtype         
---  ------                                                                        --------------  -----         
 0   DisclosureNumber                                                              92954 non-null  float64       
 1   DateCode                                                                      92954 non-null  str           
 2   Date                                                                          92956 non-null  datetime64[us]
 3   SecuritiesCode                                                                92954 non-null  string        
 4   DisclosedDate                                                                 92954 non-null  datetime64[us]
 5   DisclosedTime                                                                 92954 non-null  str  

In [8]:
REQUIRED_PRICES_COLUMNS = ['Date', 'SecuritiesCode', 'Open', 'High', 'Low', 'Close',
       'Volume', 'AdjustmentFactor', 'ExpectedDividend', 'SupervisionFlag',
       'Target']

REQUIRED_FINANCIALS_COLUMNS = ['Date', 'SecuritiesCode']

REQUIRED_STOCK_LIST_COLUMNS = ['SecuritiesCode']

'''
def require_columns(frame, required_columns):
       missing = require_columns.difference(frame.columns)
       
       if missing:
              raise ValueError(f"{frame} is missing columns: {sorted(missing)}")

require_columns(prices, REQUIRED_PRICES_COLUMNS)
'''



'\ndef require_columns(frame, required_columns):\n       missing = require_columns.difference(frame.columns)\n\n       if missing:\n              raise ValueError(f"{frame} is missing columns: {sorted(missing)}")\n\nrequire_columns(prices, REQUIRED_PRICES_COLUMNS)\n'

In [10]:


print("SHAPE")
print("Stock prices: ", prices_raw.shape)
print("stock list (metadata): ", stock_list_raw.shape)
print("financials statements: ", financials_raw.shape)

SHAPE
Stock prices:  (2332531, 12)
stock list (metadata):  (4417, 16)
financials statements:  (92956, 45)


# Audit before looking at models

Date + SecuritiesCode must uniquely identify a price row.

In [11]:
key_audit = pd.Series({
    "price_duplicate_date_security": int(prices_raw.duplicated(["Date", "SecuritiesCode"]).sum()),
    "price_missing_security": int(prices_raw["SecuritiesCode"].isna().sum()),
    "stock_list_duplicate_security": int(stock_list_raw.duplicated("SecuritiesCode").sum()),
    "financial_missing_security": int(financials_raw["SecuritiesCode"].isna().sum()),
    "financial_missing_disclosed_date": int(financials_raw["DisclosedDate"].isna().sum()),
}, name="count")
display(key_audit)

assert key_audit["price_duplicate_date_security"] == 0
assert key_audit["price_missing_security"] == 0
assert key_audit["stock_list_duplicate_security"] == 0

price_duplicate_date_security       0
price_missing_security              0
stock_list_duplicate_security       0
financial_missing_security          2
financial_missing_disclosed_date    2
Name: count, dtype: int64

In [20]:
financials_raw[financials_raw['SecuritiesCode'].isna()]

,DisclosureNumber,DateCode,Date,SecuritiesCode,DisclosedDate,DisclosedTime,DisclosedUnixTime,TypeOfDocument,CurrentPeriodEndDate,TypeOfCurrentPeriod,...,ForecastEarningsPerShare,ApplyingOfSpecificAccountingOfTheQuarterlyFinancialStatements,MaterialChangesInSubsidiaries,ChangesBasedOnRevisionsOfAccountingStandard,ChangesOtherThanOnesBasedOnRevisionsOfAccountingStandard,ChangesInAccountingEstimates,RetrospectiveRestatement,NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock,NumberOfTreasuryStockAtTheEndOfFiscalYear,AverageNumberOfShares
22557,NaN,NaN,2018-02-21,<NA>,NaT,NaN,NaN,NaN,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54934,NaN,NaN,2019-12-30,<NA>,NaT,NaN,NaN,NaN,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
financials = financials_raw[~financials_raw['SecuritiesCode'].isna()]
len(financials_raw) - len(financials)

2

In [12]:
def missingness(frame: pd.DataFrame) -> pd.DataFrame:
    return (
        pd.DataFrame({
            "missing_count": frame.isna().sum(),
            "missing_rate": frame.isna().mean(),
            "dtype": frame.dtypes.astype(str),
        })
        .sort_values("missing_rate", ascending=False)
    )

display(missingness(prices_raw).head(20))
display(missingness(financials_raw).head(25))

,missing_count,missing_rate,dtype
ExpectedDividend,2313666,0.991912,float64
Open,7608,0.003262,float64
High,7608,0.003262,float64
Close,7608,0.003262,float64
Low,7608,0.003262,float64
Target,238,0.000102,float64
RowId,0,0.000000,str
Date,0,0.000000,datetime64[us]
SecuritiesCode,0,0.000000,string
Volume,0,0.000000,int64


,missing_count,missing_rate,dtype
ApplyingOfSpecificAccountingOfTheQuarterlyFinancialStatements,85707,0.922017,object
ForecastDividendPerShare1stQuarter,73715,0.793010,str
ResultDividendPerShareAnnual,73541,0.791138,str
ResultDividendPerShareFiscalYearEnd,73540,0.791127,str
BookValuePerShare,57183,0.615162,str
ResultDividendPerShare3rdQuarter,55279,0.594679,str
ForecastDividendPerShare2ndQuarter,50337,0.541514,str
ResultDividendPerShare2ndQuarter,37016,0.398210,str
ForecastDividendPerShare3rdQuarter,32149,0.345852,str
MaterialChangesInSubsidiaries,28452,0.306080,object


In [13]:
ohlc = ["Open", "High", "Low", "Close"]
all_ohlc_missing = prices_raw[ohlc].isna().all(axis=1)
no_trade_candidate = all_ohlc_missing & prices_raw["Volume"].eq(0)

missing_price_audit = pd.Series({
    "rows": len(prices_raw),
    "all_ohlc_missing": int(all_ohlc_missing.sum()),
    "all_ohlc_missing_and_zero_volume": int(no_trade_candidate.sum()),
    "no_trade_candidate_with_target": int(prices_raw.loc[no_trade_candidate, "Target"].notna().sum()),
    "missing_target": int(prices_raw["Target"].isna().sum()),
    "negative_volume": int(prices_raw["Volume"].lt(0).sum()),
    "non_positive_adjustment_factor": int(prices_raw["AdjustmentFactor"].le(0).sum()),
}, name="count")
display(missing_price_audit)

rows                                2332531
all_ohlc_missing                       7608
all_ohlc_missing_and_zero_volume       7608
no_trade_candidate_with_target         7370
missing_target                          238
negative_volume                           0
non_positive_adjustment_factor            0
Name: count, dtype: int64

## Cleaning Overview


- **Train:** 2017–2019.
- **Validation:** 2020
- **Final test:** 2021
- **NOTE:** remove the final two trading dates before each held-out period,
  because a label near the boundary depends on future prices.
- **Primary metric:** JPX-style top-200 minus bottom-200 spread Sharpe.
- **Secondary metric:** mean daily Spearman rank correlation.
- **Diagnostics:** MAE, RMSE, and R².



In [14]:
yearly_coverage = (
    prices_raw.assign(Year=prices_raw["Date"].dt.year)
    .groupby("Year")
    .agg(
        rows=("SecuritiesCode", "size"),
        dates=("Date", "nunique"),
        securities=("SecuritiesCode", "nunique"),
        labeled_rows=("Target", "count"),
    )
)
display(yearly_coverage)

,rows,dates,securities,labeled_rows
Year,,,,
2017,463722,247,1896,463487
2018,467672,245,1930,467671
2019,468295,241,1966,468295
2020,480842,243,2000,480840
2021,452000,226,2000,452000


### Cleaning prices

In [16]:
prices_raw.columns

Index(['RowId', 'Date', 'SecuritiesCode', 'Open', 'High', 'Low', 'Close',
       'Volume', 'AdjustmentFactor', 'ExpectedDividend', 'SupervisionFlag',
       'Target'],
      dtype='str')

In [ ]:
def clean_prices(
        df: pd.DataFrame
) -> tuple[pd.DataFrame, dict]:
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'], errors="raise")

    numeric_cols = ['Open', 'High', 'Low', 'Close','Volume', 'AdjustmentFactor','Target']

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    audit = {
        "input_rows": len(df)
        ,"duplicates_stock_dates": int(df.duplicated(['Date','SecuritiesCode']).sum())
        ,"missing_close": int(df['Close'].isna().sum())
        ,"missing_volume": int(df['Volume'].isna().sum())
        ,"negative_volume": int((df['Volume'] < 0).sum())
             }

    if audit['duplicates_stock_dates']:
        duplicates = df.loc[df.duplicated(['Date','SecuritiesCode'],keep=False)].sort_values(by=['Date','SecuritiesCode'])

        display(duplicates.head(20))
        raise ValueError(
            "Duplicate Date-SecuritiesCode records found. "
            "Investigate before continuing"
        )

    complete_ohlc = df[['Open', 'High', 'Low', 'Close']].notna().all(axis=1)
    invalid_ohlc = complete_ohlc & (
        (df['High'] < df['Low'])
        | (df['High'] < df['Open'])
        | (df['High'] < df['Close'])
        | (df['Low'] > df['Open'])
        | (df['Low'] > df['Close'])
        | (df['Close'] <= 0)
        | (df['Open'] <= 0)
    )

    audit['invalid_ohcl'] = int(invalid_ohlc.sum())

    if invalid_ohlc.any():
        print(
            f"Removing {invalid_ohlc.sum():,} rows with internally inconsistent OHLC values.")

        df = df.loc[~invalid_ohlc].copy()

    invalid_volume = df['Volume'].isna() | df['Volume'] <0
    audit['invalid_volume_rows'] = int(invalid_volume.sum())

    df = df.loc[~invalid_volume].copy()

    missing_adjustment = df['AdjustmentFactor'].isna()
    audit['missing_adjustment'] = int(missing_adjustment.sum())

    print("Fill missing adjustment with 1.0 (unchanged stock split)")
    df.loc[missing_adjustment, 'AdjustmentFactor'] = 1.0

    if (df['AdjustmentFactor'] <= 0).any():
        raise ValueError("AdjustmentFactor contains non-positive values.")

    df = df.sort_values(['Date','SecuritiesCode']).reset_index(drop=True)

    audit['output_rows'] = len(df)
    audit['removed_rows'] = audit['output_rows'] - audit['input_rows']

    return df, audit

In [19]:
prices, prices_audit = clean_prices(
    prices_raw
)
display(prices_audit)

assert len(prices) == len(prices_raw)
assert not prices.duplicated(["Date", "SecuritiesCode"]).any()

Fill missing adjustment with 1.0 (unchanged stock split)


{'input_rows': 2332531,
 'duplicates_stock_dates': 0,
 'missing_close': 7608,
 'missing_volume': 0,
 'negative_volume': 0,
 'invalid_ohcl': 0,
 'invalid_volume_rows': 0,
 'missing_adjustment': 0,
 'output_rows': 2332531,
 'removed_rows': 0}

In [269]:
prices.head()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,NaN,False,0.000730
1,20170104_1332,2017-01-04,1332,568.0,576.0,563.0,571.0,2798500,1.0,NaN,False,0.012324
2,20170104_1333,2017-01-04,1333,3150.0,3210.0,3140.0,3210.0,270800,1.0,NaN,False,0.006154
3,20170104_1376,2017-01-04,1376,1510.0,1550.0,1510.0,1550.0,11300,1.0,NaN,False,0.011053
4,20170104_1377,2017-01-04,1377,3270.0,3350.0,3270.0,3330.0,150800,1.0,NaN,False,0.003026


In [270]:
prices.info()
#date column is in str, need conversion to datetime
#maybe no need for RowId, aldready contained in Date and SecuritiesCode columns

<class 'pandas.DataFrame'>
RangeIndex: 2332531 entries, 0 to 2332530
Data columns (total 12 columns):
 #   Column            Dtype         
---  ------            -----         
 0   RowId             str           
 1   Date              datetime64[us]
 2   SecuritiesCode    string        
 3   Open              float64       
 4   High              float64       
 5   Low               float64       
 6   Close             float64       
 7   Volume            int64         
 8   AdjustmentFactor  float64       
 9   ExpectedDividend  float64       
 10  SupervisionFlag   bool          
 11  Target            float64       
dtypes: bool(1), datetime64[us](1), float64(7), int64(1), str(1), string(1)
memory usage: 198.0 MB


In [271]:
print(prices['SecuritiesCode'].unique().shape)
print(stock_list['SecuritiesCode'].unique().shape)
print(financials['SecuritiesCode'].unique().shape)

print(prices['SecuritiesCode'].dtype)
print(stock_list['SecuritiesCode'].dtype)
print(financials['SecuritiesCode'].dtype)

(2000,)
(4417,)
(4072,)
string
string
string


In [272]:
price_key_duplicates = prices.duplicated(['Date','SecuritiesCode']).sum()
print('Duplicate price keys: ', price_key_duplicates)

print("Non positive volumn: ",prices['Volume'].lt(0).sum())
print("Non positive adjustment factor: ", prices['AdjustmentFactor'].lt(0).sum())
print("Missing target: ", prices['Target'].isna().sum())

Duplicate price keys:  0
Non positive volumn:  0
Non positive adjustment factor:  0
Missing target:  238


In [273]:
codeMissingTargets = prices[prices['Target'].isna()]['SecuritiesCode'].unique().tolist()
codeMissingTargets

['3540', '4382', '4056', '2987']

In [274]:
prices[prices['SecuritiesCode'].isin(codeMissingTargets)]

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target
401,20170104_3540,2017-01-04,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
2266,20170105_3540,2017-01-05,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
4131,20170106_3540,2017-01-06,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
5996,20170110_3540,2017-01-10,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
7861,20170111_3540,2017-01-11,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2329155,20211202_4382,2021-12-02,4382,1440.0,1464.0,1389.0,1409.0,108800,1.0,NaN,False,-0.007503
2330831,20211203_2987,2021-12-03,2987,3085.0,3515.0,3030.0,3460.0,149700,1.0,NaN,False,-0.050228
2330948,20211203_3540,2021-12-03,3540,4850.0,5020.0,4830.0,4980.0,27700,1.0,NaN,False,0.021063
2331075,20211203_4056,2021-12-03,4056,1714.0,1800.0,1680.0,1797.0,86000,1.0,NaN,False,0.055843


In [275]:
prices[prices['SecuritiesCode']== '3540']

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target
401,20170104_3540,2017-01-04,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
2266,20170105_3540,2017-01-05,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
4131,20170106_3540,2017-01-06,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
5996,20170110_3540,2017-01-10,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
7861,20170111_3540,2017-01-11,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2322948,20211129_3540,2021-11-29,3540,5000.0,5130.0,4980.0,4980.0,13000,1.0,NaN,False,0.055966
2324948,20211130_3540,2021-11-30,3540,5010.0,5050.0,4725.0,4735.0,21800,1.0,NaN,False,-0.026000
2326948,20211201_3540,2021-12-01,3540,4735.0,5040.0,4710.0,5000.0,27300,1.0,NaN,False,0.022587
2328948,20211202_3540,2021-12-02,3540,4900.0,4985.0,4855.0,4870.0,12300,1.0,NaN,False,0.001004


In [276]:
stock_list[stock_list['SecuritiesCode'] == '3540']

,SecuritiesCode,EffectiveDate,Name,Section/Products,NewMarketSegment,33SectorCode,33SectorName,17SectorCode,17SectorName,NewIndexSeriesSizeCode,NewIndexSeriesSize,TradeDate,Close,IssuedShares,MarketCapitalization,Universe0
1178,3540,1970-01-01 00:00:00.020211230,"C.I.MEDICAL CO.,LTD.",JASDAQ(Standard / Domestic),Standard Market,6050,Wholesale Trade,13,COMMERCIAL & WHOLESALE TRADE,-,-,1970-01-01 00:00:00.020211230,4635.0,10000000.0,4.635000e+10,True


In [277]:
prices.isna().sum()

RowId                     0
Date                      0
SecuritiesCode            0
Open                   7608
High                   7608
Low                    7608
Close                  7608
Volume                    0
AdjustmentFactor          0
ExpectedDividend    2313666
SupervisionFlag           0
Target                  238
dtype: int64

In [278]:
int(prices.duplicated(['Date']).sum())

2331329

In [280]:
prices, prices_audit = clean_prices(df=prices)
pd.Series(prices_audit, name="value")

Fill missing adjustment with 1.0 (unchanged stock split)


input_rows                2332531
duplicates_stock_dates          0
missing_close                7608
missing_volume                  0
negative_volume                 0
invalid_ohcl                    0
invalid_volume_rows             0
missing_adjustment              0
output_rows               2332531
removed_rows                    0
Name: value, dtype: int64

In [281]:
prices.isna().sum()

RowId                     0
Date                      0
SecuritiesCode            0
Open                   7608
High                   7608
Low                    7608
Close                  7608
Volume                    0
AdjustmentFactor          0
ExpectedDividend    2313666
SupervisionFlag           0
Target                  238
dtype: int64

In [282]:
prices[prices['Close'].isna()]

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target
401,20170104_3540,2017-01-04,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
1753,20170104_9539,2017-01-04,9539,NaN,NaN,NaN,NaN,0,1.0,NaN,False,-0.004149
2266,20170105_3540,2017-01-05,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
2511,20170105_4621,2017-01-05,4621,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.000000
4131,20170106_3540,2017-01-06,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2330563,20211203_1787,2021-12-03,1787,NaN,NaN,NaN,NaN,0,1.0,NaN,False,-0.030351
2330786,20211203_2761,2021-12-03,2761,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.000000
2331453,20211203_5918,2021-12-03,5918,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.015625
2332336,20211203_9083,2021-12-03,9083,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.009615


In [283]:
ohlc =  ['Open', 'High', 'Low', 'Close']
missing_close = prices['Close'].isna()
missing_close_audit = pd.Series({
    'missing_close_rows': int(missing_close.sum())
    ,'all_ohlc_missing': int(prices.loc[missing_close, ohlc].isna().all(axis=1).sum())
    ,'volume_zero': int(prices.loc[missing_close,'Volume'].eq(0).sum())
    ,'target_available': int(prices.loc[missing_close,'Target'].notna().sum())
    ,'target_missing': int(prices.loc[missing_close,'Target'].isna().sum())
    ,'affected_stocks': int(prices.loc[missing_close, 'SecuritiesCode'].nunique())
})
missing_close_audit

missing_close_rows    7608
all_ohlc_missing      7608
volume_zero           7608
target_available      7370
target_missing         238
affected_stocks       1992
dtype: int64

In [284]:
prices.loc[missing_close].groupby('Date').size().sort_values(ascending=False).head(20)

Date
2020-10-01    1988
2017-03-16      15
2019-04-04      14
2019-10-09      14
2021-10-29      13
2018-03-07      12
2017-03-30      12
2020-07-16      12
2020-09-17      12
2020-05-14      12
2017-02-07      12
2021-08-05      12
2017-01-20      12
2021-11-11      12
2018-08-23      12
2018-07-11      11
2019-03-28      11
2020-10-19      11
2018-09-06      11
2018-08-16      11
dtype: int64

In [285]:
daily_status = (
    prices.groupby("Date")
    .agg(total_rows=("SecuritiesCode", "size"),missing_close_rows=("Close",lambda values: values.isna().sum())
    ,target_available=("Target","count"))
)

daily_status["missing_close_rate"] = (
    daily_status["missing_close_rows"]
    / daily_status["total_rows"]
)

daily_status.sort_values(by='missing_close_rate', ascending=False)

,total_rows,missing_close_rows,target_available,missing_close_rate
Date,,,,
2020-10-01,1988,1988,1988,1.000000
2017-03-16,1867,15,1866,0.008034
2019-04-04,1938,14,1938,0.007224
2019-10-09,1951,14,1951,0.007176
2021-10-29,2000,13,2000,0.006500
...,...,...,...,...
2021-05-06,2000,0,2000,0.000000
2020-03-24,1971,0,1971,0.000000
2020-03-19,1971,0,1971,0.000000


In [286]:
all_ohlc_missing = prices[ohlc].isna().all(axis=1)
prices['NoTradeFlag'] = (all_ohlc_missing & prices['Volume'].eq(0)).astype("int8")
prices.head()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,NaN,False,0.000730,0
1,20170104_1332,2017-01-04,1332,568.0,576.0,563.0,571.0,2798500,1.0,NaN,False,0.012324,0
2,20170104_1333,2017-01-04,1333,3150.0,3210.0,3140.0,3210.0,270800,1.0,NaN,False,0.006154,0
3,20170104_1376,2017-01-04,1376,1510.0,1550.0,1510.0,1550.0,11300,1.0,NaN,False,0.011053,0
4,20170104_1377,2017-01-04,1377,3270.0,3350.0,3270.0,3330.0,150800,1.0,NaN,False,0.003026,0


In [287]:
daily_market_status = (
    prices.groupby("Date")
    .agg(total_rows=("SecuritiesCode", "size")
    ,no_trade_securitiescode =('NoTradeFlag','sum')
    ,target_available=("Target","count"))
)

daily_market_status["no_trade_rate"] = (
    daily_market_status["no_trade_securitiescode"]
    / daily_status["total_rows"]
)

daily_market_status.sort_values(by='no_trade_rate', ascending=False)

,total_rows,no_trade_securitiescode,target_available,no_trade_rate
Date,,,,
2020-10-01,1988,1988,1988,1.000000
2017-03-16,1867,15,1866,0.008034
2019-04-04,1938,14,1938,0.007224
2019-10-09,1951,14,1951,0.007176
2021-10-29,2000,13,2000,0.006500
...,...,...,...,...
2021-05-06,2000,0,2000,0.000000
2020-03-24,1971,0,1971,0.000000
2020-03-19,1971,0,1971,0.000000


In [288]:
daily_market_status['MarketWideNoTradeFlag'] = daily_market_status['no_trade_rate'] >= 0.9
market_status_by_date = daily_market_status['MarketWideNoTradeFlag']
market_status_by_date

Date
2017-01-04    False
2017-01-05    False
2017-01-06    False
2017-01-10    False
2017-01-11    False
              ...  
2021-11-29    False
2021-11-30    False
2021-12-01    False
2021-12-02    False
2021-12-03    False
Name: MarketWideNoTradeFlag, Length: 1202, dtype: bool

In [289]:
market_status_by_date['2020-10-01']

np.True_

In [290]:
prices['MarketWideNoTradeFlag'] = prices['Date'].map(market_status_by_date).fillna(False).astype("int8")
prices.head()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWideNoTradeFlag
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,NaN,False,0.000730,0,0
1,20170104_1332,2017-01-04,1332,568.0,576.0,563.0,571.0,2798500,1.0,NaN,False,0.012324,0,0
2,20170104_1333,2017-01-04,1333,3150.0,3210.0,3140.0,3210.0,270800,1.0,NaN,False,0.006154,0,0
3,20170104_1376,2017-01-04,1376,1510.0,1550.0,1510.0,1550.0,11300,1.0,NaN,False,0.011053,0,0
4,20170104_1377,2017-01-04,1377,3270.0,3350.0,3270.0,3330.0,150800,1.0,NaN,False,0.003026,0,0


In [291]:
prices[prices['Date'] == '2020-10-01']

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWideNoTradeFlag
1755040,20201001_1301,2020-10-01,1301,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.029208,1,1
1755041,20201001_1332,2020-10-01,1332,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.027211,1,1
1755042,20201001_1333,2020-10-01,1333,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.027695,1,1
1755043,20201001_1375,2020-10-01,1375,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.023833,1,1
1755044,20201001_1376,2020-10-01,1376,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.022152,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1757023,20201001_9990,2020-10-01,9990,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.023297,1,1
1757024,20201001_9991,2020-10-01,9991,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.041621,1,1
1757025,20201001_9993,2020-10-01,9993,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.034006,1,1
1757026,20201001_9994,2020-10-01,9994,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.025047,1,1


In [292]:
prices['StockSpecificNoTradeFlag'] = (prices['NoTradeFlag'].eq(1) & prices['MarketWideNoTradeFlag'].eq(0)).astype('int8')
prices[prices['StockSpecificNoTradeFlag']>0]

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWideNoTradeFlag,StockSpecificNoTradeFlag
401,20170104_3540,2017-01-04,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN,1,0,1
1753,20170104_9539,2017-01-04,9539,NaN,NaN,NaN,NaN,0,1.0,NaN,False,-0.004149,1,0,1
2266,20170105_3540,2017-01-05,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN,1,0,1
2511,20170105_4621,2017-01-05,4621,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.000000,1,0,1
4131,20170106_3540,2017-01-06,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2330563,20211203_1787,2021-12-03,1787,NaN,NaN,NaN,NaN,0,1.0,NaN,False,-0.030351,1,0,1
2330786,20211203_2761,2021-12-03,2761,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.000000,1,0,1
2331453,20211203_5918,2021-12-03,5918,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.015625,1,0,1
2332336,20211203_9083,2021-12-03,9083,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.009615,1,0,1


In [293]:
prices[
    [
        "NoTradeFlag",
        "MarketWideNoTradeFlag",
        "StockSpecificNoTradeFlag",
    ]
].sum()

NoTradeFlag                 7608
MarketWideNoTradeFlag       1988
StockSpecificNoTradeFlag    5620
dtype: int64

In [294]:
prices.corr()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWideNoTradeFlag,StockSpecificNoTradeFlag
RowId,1.000000,0.984664,-0.002366,0.027686,0.028284,0.026852,0.027525,-0.020065,-0.010067,0.059735,0.003972,-0.002972,0.011332,0.021572,0.000339
Date,0.984664,1.000000,-0.002450,0.030181,0.030733,0.029383,0.030009,-0.021693,-0.008811,0.043939,0.004641,-0.001507,0.013411,0.025766,0.000260
SecuritiesCode,-0.002366,-0.002450,1.000000,0.016788,0.016348,0.017235,0.016798,0.040869,0.001716,0.033389,-0.001591,-0.003389,-0.005367,-0.000129,-0.006165
Open,0.027686,0.030181,0.016788,1.000000,0.999851,0.999854,0.999730,-0.036524,-0.006918,0.558148,-0.004112,-0.003919,NaN,NaN,NaN
High,0.028284,0.030733,0.016348,0.999851,1.000000,0.999756,0.999857,-0.036316,-0.006933,0.557004,-0.004114,-0.003809,NaN,NaN,NaN
Low,0.026852,0.029383,0.017235,0.999854,0.999756,1.000000,0.999862,-0.036639,-0.006893,0.558808,-0.004092,-0.003958,NaN,NaN,NaN
Close,0.027525,0.030009,0.016798,0.999730,0.999857,0.999862,1.000000,-0.036473,-0.006907,0.558037,-0.004101,-0.003852,NaN,NaN,NaN
Volume,-0.020065,-0.021693,0.040869,-0.036524,-0.036316,-0.036639,-0.036473,1.000000,0.005304,-0.019052,0.101938,-0.000873,-0.010120,-0.005167,-0.008694
AdjustmentFactor,-0.010067,-0.008811,0.001716,-0.006918,-0.006933,-0.006893,-0.006907,0.005304,1.000000,NaN,-0.000113,-0.000091,-0.000073,-0.000219,0.000045
ExpectedDividend,0.059735,0.043939,0.033389,0.558148,0.557004,0.558808,0.558037,-0.019052,NaN,1.000000,-0.006183,-0.148628,0.003139,NaN,0.003139


In [295]:
prices.isna().sum()

RowId                             0
Date                              0
SecuritiesCode                    0
Open                           7608
High                           7608
Low                            7608
Close                          7608
Volume                            0
AdjustmentFactor                  0
ExpectedDividend            2313666
SupervisionFlag                   0
Target                          238
NoTradeFlag                       0
MarketWideNoTradeFlag             0
StockSpecificNoTradeFlag          0
dtype: int64

In [296]:
n_before =len(prices)
print("Length before exlcuding market with no trade: ", n_before)

prices = prices.loc[prices['MarketWideNoTradeFlag'].eq(0)].copy()
print(f"Length after: {len(prices)},{len(prices)-n_before} obs")

Length before exlcuding market with no trade:  2332531
Length after: 2330543,-1988 obs


In [297]:
prices.isna().sum()

RowId                             0
Date                              0
SecuritiesCode                    0
Open                           5620
High                           5620
Low                            5620
Close                          5620
Volume                            0
AdjustmentFactor                  0
ExpectedDividend            2311678
SupervisionFlag                   0
Target                          238
NoTradeFlag                       0
MarketWideNoTradeFlag             0
StockSpecificNoTradeFlag          0
dtype: int64

In [298]:
prices['HasExpectedDividend'] = prices['ExpectedDividend'].notna().astype('int8')
prices[prices['HasExpectedDividend']==1]

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWideNoTradeFlag,StockSpecificNoTradeFlag,HasExpectedDividend
13269,20170116_2590,2017-01-16,2590,6060.0,6130.0,6060.0,6060.0,125800,1.0,30.0,False,0.005051,0,0,0,1
13726,20170116_4699,2017-01-16,4699,1119.0,1119.0,1081.0,1087.0,11500,1.0,0.0,False,-0.032833,0,0,0,1
26179,20170125_1928,2017-01-25,1928,1886.0,1894.5,1865.0,1871.5,3310500,1.0,32.0,False,-0.018336,0,0,0,1
26249,20170125_2217,2017-01-25,2217,515.0,517.0,512.0,512.0,162000,1.0,4.0,False,0.016000,0,0,0,1
26281,20170125_2353,2017-01-25,2353,153.0,154.0,151.0,152.0,431700,1.0,0.0,False,-0.006452,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2320290,20211125_8923,2021-11-25,8923,1047.0,1051.0,1030.0,1030.0,389700,1.0,38.0,False,-0.073099,0,0,0,1
2320357,20211125_9278,2021-11-25,9278,995.0,995.0,989.0,990.0,12800,1.0,0.0,False,0.000000,0,0,0,1
2320370,20211125_9369,2021-11-25,9369,2000.0,2009.0,1989.0,1990.0,69100,1.0,23.0,False,-0.057692,0,0,0,1
2320456,20211125_9717,2021-11-25,9717,1258.0,1260.0,1222.0,1223.0,313200,1.0,50.0,False,-0.127517,0,0,0,1


In [299]:
prices['ExpectedDividend'] = prices['ExpectedDividend'].fillna(0.0)

In [300]:
prices.isna().sum()

RowId                          0
Date                           0
SecuritiesCode                 0
Open                        5620
High                        5620
Low                         5620
Close                       5620
Volume                         0
AdjustmentFactor               0
ExpectedDividend               0
SupervisionFlag                0
Target                       238
NoTradeFlag                    0
MarketWideNoTradeFlag          0
StockSpecificNoTradeFlag       0
HasExpectedDividend            0
dtype: int64

In [301]:
prices = prices.sort_values(['Date','SecuritiesCode']).reset_index(drop=True)
prices.head()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWideNoTradeFlag,StockSpecificNoTradeFlag,HasExpectedDividend
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,0.0,False,0.000730,0,0,0,0
1,20170104_1332,2017-01-04,1332,568.0,576.0,563.0,571.0,2798500,1.0,0.0,False,0.012324,0,0,0,0
2,20170104_1333,2017-01-04,1333,3150.0,3210.0,3140.0,3210.0,270800,1.0,0.0,False,0.006154,0,0,0,0
3,20170104_1376,2017-01-04,1376,1510.0,1550.0,1510.0,1550.0,11300,1.0,0.0,False,0.011053,0,0,0,0
4,20170104_1377,2017-01-04,1377,3270.0,3350.0,3270.0,3330.0,150800,1.0,0.0,False,0.003026,0,0,0,0


In [302]:
prices['LastTradeDate'] = prices['Date'].where(prices['Close'].notna())

prices['LastTradeDate'] = prices.groupby(by='SecuritiesCode',sort=False)['LastTradeDate'].ffill()

prices['DaysSinceLastTrade'] = (prices['Date'] - prices['LastTradeDate']).dt.days
prices.head()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWideNoTradeFlag,StockSpecificNoTradeFlag,HasExpectedDividend,LastTradeDate,DaysSinceLastTrade
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,0.0,False,0.000730,0,0,0,0,2017-01-04,0.0
1,20170104_1332,2017-01-04,1332,568.0,576.0,563.0,571.0,2798500,1.0,0.0,False,0.012324,0,0,0,0,2017-01-04,0.0
2,20170104_1333,2017-01-04,1333,3150.0,3210.0,3140.0,3210.0,270800,1.0,0.0,False,0.006154,0,0,0,0,2017-01-04,0.0
3,20170104_1376,2017-01-04,1376,1510.0,1550.0,1510.0,1550.0,11300,1.0,0.0,False,0.011053,0,0,0,0,2017-01-04,0.0
4,20170104_1377,2017-01-04,1377,3270.0,3350.0,3270.0,3330.0,150800,1.0,0.0,False,0.003026,0,0,0,0,2017-01-04,0.0


In [303]:
prices['CumulativeAdjustment'] = prices.groupby(by='SecuritiesCode')[
    'AdjustmentFactor'].transform(lambda values: (values.fillna(1.0).iloc[::-1].cumprod().iloc[::-1]))

prices['AdjustedClose'] = prices['Close'] * prices['CumulativeAdjustment']

In [304]:
grouped_close = prices.groupby(by='SecuritiesCode')['AdjustedClose']
prices['CloseLag1'] = grouped_close.shift(periods=1)
prices['CloseLag5'] = grouped_close.shift(periods=5)
prices['CloseLag20'] = grouped_close.shift(periods=20)
prices['CloseLag60'] = grouped_close.shift(periods=60)
prices.head()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWideNoTradeFlag,StockSpecificNoTradeFlag,HasExpectedDividend,LastTradeDate,DaysSinceLastTrade,CumulativeAdjustment,AdjustedClose,CloseLag1,CloseLag5,CloseLag20,CloseLag60
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,0.0,False,0.000730,0,0,0,0,2017-01-04,0.0,1.0,2742.0,NaN,NaN,NaN,NaN
1,20170104_1332,2017-01-04,1332,568.0,576.0,563.0,571.0,2798500,1.0,0.0,False,0.012324,0,0,0,0,2017-01-04,0.0,1.0,571.0,NaN,NaN,NaN,NaN
2,20170104_1333,2017-01-04,1333,3150.0,3210.0,3140.0,3210.0,270800,1.0,0.0,False,0.006154,0,0,0,0,2017-01-04,0.0,1.0,3210.0,NaN,NaN,NaN,NaN
3,20170104_1376,2017-01-04,1376,1510.0,1550.0,1510.0,1550.0,11300,1.0,0.0,False,0.011053,0,0,0,0,2017-01-04,0.0,1.0,1550.0,NaN,NaN,NaN,NaN
4,20170104_1377,2017-01-04,1377,3270.0,3350.0,3270.0,3330.0,150800,1.0,0.0,False,0.003026,0,0,0,0,2017-01-04,0.0,1.0,3330.0,NaN,NaN,NaN,NaN


In [305]:
sorting_keys = pd.MultiIndex.from_frame(prices[['Date','SecuritiesCode']])
print("correctly sorted", sorting_keys.is_monotonic_increasing)

correctly sorted True


In [306]:
prices['LogReturn1Day'] = np.log(prices['AdjustedClose'] / prices['CloseLag1'])

prices['Momentum5Days'] = prices['AdjustedClose'] / prices['CloseLag5'] - 1
prices['Momentum20Days'] = prices['AdjustedClose'] / prices['CloseLag20'] - 1
prices['Momentum60Days'] = prices['AdjustedClose'] / prices['CloseLag60'] - 1

prices['MovingAverage20Days'] = grouped_close.transform(lambda values: values.rolling(window=20, min_periods=15).mean())
prices['ClosetoMovingAverage20Days'] = prices['AdjustedClose'] / prices['MovingAverage20Days'] - 1 

In [307]:
META_CANDIDATES = [
    'SecuritiesCode'
    ,'33SectorCode'
    ,'17SectorCode'
    ,'NewMarketSegment'
]

meta_columns = [column for column in META_CANDIDATES if column in stock_list.columns]
stock_meta = stock_list[meta_columns].copy()

duplicate_meta = stock_meta.duplicated('SecuritiesCode').copy()
if duplicate_meta.any():
    display(stock_meta.loc[duplicate_meta].sort_values('SecuritiesCode').head(20))
    raise ValueError("stock_list has multiple rows per security code")

for column in meta_columns:
    if column != 'SecuritiesCode':
        stock_meta[column] = stock_meta[column].astype("string").fillna("__MISSING__")

In [308]:
prices.columns

Index(['RowId', 'Date', 'SecuritiesCode', 'Open', 'High', 'Low', 'Close',
       'Volume', 'AdjustmentFactor', 'ExpectedDividend', 'SupervisionFlag',
       'Target', 'NoTradeFlag', 'MarketWideNoTradeFlag',
       'StockSpecificNoTradeFlag', 'HasExpectedDividend', 'LastTradeDate',
       'DaysSinceLastTrade', 'CumulativeAdjustment', 'AdjustedClose',
       'CloseLag1', 'CloseLag5', 'CloseLag20', 'CloseLag60', 'LogReturn1Day',
       'Momentum5Days', 'Momentum20Days', 'Momentum60Days',
       'MovingAverage20Days', 'ClosetoMovingAverage20Days'],
      dtype='str')

In [309]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [310]:
prices.corr()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWideNoTradeFlag,StockSpecificNoTradeFlag,HasExpectedDividend,LastTradeDate,DaysSinceLastTrade,CumulativeAdjustment,AdjustedClose,CloseLag1,CloseLag5,CloseLag20,CloseLag60,LogReturn1Day,Momentum5Days,Momentum20Days,Momentum60Days,MovingAverage20Days,ClosetoMovingAverage20Days
RowId,1.000000,0.984664,-0.002364,0.027686,0.028284,0.026852,0.027525,-0.019959,-0.010064,0.004421,0.003982,-0.003556,0.000370,NaN,0.000370,0.000175,0.984663,0.005028,-0.142275,0.033513,0.033468,0.033295,0.032250,0.027967,-0.005250,-0.007636,-0.014405,-0.010603,0.033129,-0.017901
Date,0.984664,1.000000,-0.002449,0.030181,0.030733,0.029383,0.030009,-0.021567,-0.008809,0.003635,0.004654,-0.002203,0.000297,NaN,0.000297,0.000783,1.000000,0.004946,-0.147353,0.035713,0.035637,0.035392,0.033477,0.027379,-0.004398,-0.008067,-0.006939,0.005136,0.034946,-0.013594
SecuritiesCode,-0.002364,-0.002449,1.000000,0.016788,0.016348,0.017235,0.016798,0.040886,0.001717,0.003237,-0.001592,-0.003361,-0.006168,NaN,-0.006168,0.001400,-0.002585,-0.004588,0.020521,0.026960,0.026966,0.026993,0.027130,0.027453,-0.002410,-0.007812,-0.016398,-0.027838,0.027901,-0.010765
Open,0.027686,0.030181,0.016788,1.000000,0.999851,0.999854,0.999730,-0.036524,-0.006918,0.041956,-0.004112,-0.003919,NaN,NaN,NaN,0.000805,0.030181,NaN,-0.108792,0.930306,0.932217,0.931237,0.926971,0.916340,0.002420,0.024737,0.054606,0.092732,0.933309,0.046597
High,0.028284,0.030733,0.016348,0.999851,1.000000,0.999756,0.999857,-0.036316,-0.006933,0.041875,-0.004114,-0.003809,NaN,NaN,NaN,0.000815,0.030733,NaN,-0.109066,0.930055,0.931750,0.930781,0.926510,0.915801,0.008176,0.027598,0.056581,0.094655,0.932873,0.049202
Low,0.026852,0.029383,0.017235,0.999854,0.999756,1.000000,0.999862,-0.036639,-0.006893,0.042051,-0.004092,-0.003958,NaN,NaN,NaN,0.000791,0.029383,NaN,-0.108534,0.930780,0.932438,0.931482,0.927233,0.916722,0.007614,0.026294,0.054845,0.092037,0.933549,0.047842
Close,0.027525,0.030009,0.016798,0.999730,0.999857,0.999862,1.000000,-0.036473,-0.006907,0.042064,-0.004101,-0.003852,NaN,NaN,NaN,0.000876,0.030009,NaN,-0.108808,0.930559,0.931997,0.931033,0.926776,0.916166,0.013146,0.029271,0.056874,0.093948,0.933124,0.050530
Volume,-0.019959,-0.021567,0.040886,-0.036524,-0.036316,-0.036639,-0.036473,1.000000,0.005303,-0.000502,0.101938,-0.000734,-0.008702,NaN,-0.008702,0.001767,-0.021593,-0.006310,0.194900,-0.018191,-0.018341,-0.018612,-0.019198,-0.020904,0.010679,0.022020,0.021084,0.015181,-0.018884,0.020266
AdjustmentFactor,-0.010064,-0.008809,0.001717,-0.006918,-0.006933,-0.006893,-0.006907,0.005303,1.000000,-0.000401,-0.000113,-0.000085,0.000045,NaN,0.000045,-0.000677,-0.008817,0.000614,0.058906,0.002833,0.002803,0.002770,0.002952,0.002520,0.001562,0.001905,0.002198,0.001105,0.002717,0.003450
ExpectedDividend,0.004421,0.003635,0.003237,0.041956,0.041875,0.042051,0.042064,-0.000502,-0.000401,1.000000,-0.000064,-0.035463,-0.001076,NaN,-0.001076,0.591631,0.003628,-0.000902,-0.006709,0.040743,0.040950,0.040105,0.040102,0.041816,-0.003305,0.008022,0.001929,-0.002844,0.040456,0.005062


In [311]:
prices.isna().sum()

RowId                              0
Date                               0
SecuritiesCode                     0
Open                            5620
High                            5620
Low                             5620
Close                           5620
Volume                             0
AdjustmentFactor                   0
ExpectedDividend                   0
SupervisionFlag                    0
Target                           238
NoTradeFlag                        0
MarketWideNoTradeFlag              0
StockSpecificNoTradeFlag           0
HasExpectedDividend                0
LastTradeDate                    262
DaysSinceLastTrade               262
CumulativeAdjustment               0
AdjustedClose                   5620
CloseLag1                       7615
CloseLag5                      15598
CloseLag20                     45499
CloseLag60                    125271
LogReturn1Day                  11224
Momentum5Days                  19753
Momentum20Days                 49600
M

In [313]:
prices.describe()


,Date,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,Target,NoTradeFlag,MarketWideNoTradeFlag,StockSpecificNoTradeFlag,HasExpectedDividend,LastTradeDate,DaysSinceLastTrade,CumulativeAdjustment,AdjustedClose,CloseLag1,CloseLag5,CloseLag20,CloseLag60,LogReturn1Day,Momentum5Days,Momentum20Days,Momentum60Days,MovingAverage20Days,ClosetoMovingAverage20Days
count,2330543,2.324923e+06,2.324923e+06,2.324923e+06,2.324923e+06,2.330543e+06,2.330543e+06,2.330543e+06,2.330305e+06,2.330543e+06,2330543.0,2.330543e+06,2.330543e+06,2330281,2.330281e+06,2.330543e+06,2.324923e+06,2.322928e+06,2.314945e+06,2.285044e+06,2.205272e+06,2.319319e+06,2.310790e+06,2.280943e+06,2.201173e+06,2.296303e+06,2.293255e+06
mean,2019-06-29 12:16:29.568526,2.594511e+03,2.626540e+03,2.561227e+03,2.594023e+03,6.925268e+05,1.000508e+00,1.782265e-01,4.267045e-04,2.411455e-03,0.0,2.411455e-03,8.094680e-03,2019-06-29 13:56:11.546521,5.122987e-03,1.110295e+00,2.451855e+03,2.451566e+03,2.450529e+03,2.444675e+03,2.427852e+03,1.610588e-04,2.126022e-03,9.313319e-03,2.762625e-02,2.445174e+03,2.476687e-03
min,2017-01-04 00:00:00,1.400000e+01,1.500000e+01,1.300000e+01,1.400000e+01,0.000000e+00,1.000000e-01,0.000000e+00,-5.785414e-01,0.000000e+00,0.0,0.000000e+00,0.000000e+00,2017-01-04 00:00:00,0.000000e+00,4.000000e-02,2.700000e+01,2.700000e+01,2.700000e+01,2.700000e+01,2.700000e+01,-8.640337e-01,-7.762596e-01,-8.252981e-01,-8.328373e-01,2.960000e+01,-7.298272e-01
25%,2018-04-05 00:00:00,1.022000e+03,1.035000e+03,1.009000e+03,1.022000e+03,3.040000e+04,1.000000e+00,0.000000e+00,-1.051051e-02,0.000000e+00,0.0,0.000000e+00,0.000000e+00,2018-04-05 00:00:00,0.000000e+00,1.000000e+00,1.011000e+03,1.011000e+03,1.011000e+03,1.010000e+03,1.007000e+03,-1.061018e-02,-2.356406e-02,-4.667864e-02,-7.692308e-02,1.013175e+03,-2.572491e-02
50%,2019-07-04 00:00:00,1.812000e+03,1.834000e+03,1.790000e+03,1.811000e+03,1.073000e+05,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,2019-07-04 00:00:00,0.000000e+00,1.000000e+00,1.737000e+03,1.737000e+03,1.737000e+03,1.735000e+03,1.728000e+03,0.000000e+00,4.330879e-04,3.948667e-03,8.185539e-03,1.737100e+03,1.598295e-03
75%,2020-09-25 00:00:00,3.030000e+03,3.070000e+03,2.995000e+03,3.030000e+03,4.026000e+05,1.000000e+00,0.000000e+00,1.051051e-02,0.000000e+00,0.0,0.000000e+00,0.000000e+00,2020-09-25 00:00:00,0.000000e+00,1.000000e+00,2.855000e+03,2.855000e+03,2.854000e+03,2.850000e+03,2.838000e+03,1.050568e-02,2.539405e-02,5.758427e-02,1.070645e-01,2.851550e+03,2.949014e-02
max,2021-12-03 00:00:00,1.099500e+05,1.105000e+05,1.072000e+05,1.095500e+05,6.436540e+08,2.000000e+01,1.070000e+03,1.119512e+00,1.000000e+00,0.0,1.000000e+00,1.000000e+00,2021-12-03 00:00:00,2.700000e+01,2.000000e+01,1.095500e+05,1.095500e+05,1.095500e+05,1.095500e+05,1.095500e+05,7.511860e-01,2.155702e+00,4.758621e+00,6.236923e+00,1.011950e+05,1.711324e+00
std,NaN,3.577192e+03,3.619363e+03,3.533494e+03,3.576538e+03,3.912872e+06,6.775928e-02,3.334701e+00,2.339166e-02,4.904734e-02,0.0,4.904734e-02,8.960558e-02,NaN,1.437079e-01,1.099327e+00,3.366477e+03,3.365348e+03,3.361043e+03,3.341442e+03,3.292074e+03,2.330783e-02,5.352567e-02,1.085697e-01,1.921336e-01,3.329535e+03,5.838769e-02
